### **Practical 1.8 Auditing & Mitigating a Classifier with Fairlearn**
#### **Responsible AI for Classical Systems**
In this practical you will take a real classifier and:
1. **Measure** how it performs for each group, with honest uncertainty
2. **See** the three fairness families as numbers, and watch the *impossibility result* appear
3. **Mitigate** the disparity three ways, and compare what each one buys and costs

>> **How to use this notebook**: run each cell in order with **Shift + Enter**. Every step has a plain-language explanation first, then the code. Read the explanation, run the code, look at the output, then read the "what this means" note. New to Colab? See the Google Colab guide in the Learning Resources section.

### **Step 0 - Setup**
First, install Fairlearn (the fairness toolkit) and any other required Python packages.

When using a Google Colab notebook, install Fairlearn in a notebook cell with:

```python
!pip install -q fairlearn==0.14.0
```

When using a Jupyter Notebook in VS Code, install the package from the PowerShell terminal with:

```powershell
python -m pip install fairlearn==0.14.0
```

Other required packages can be installed in the same way:

```powershell
python -m pip install numpy
python -m pip install pandas
python -m pip install matplotlib
python -m pip install scikit-learn
```

In [1]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from fairlearn.metrics import(
    MetricFrame, selection_rate, true_positive_rate, false_positive_rate, false_negative_rate, count, demographic_parity_difference, equalized_odds_difference
)
from fairlearn.reductions import(
    ExponentiatedGradient, DemographicParity, EqualizedOdds
)
from fairlearn.postprocessing import ThresholdOptimizer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup complete.")


Setup complete.


### **Step 1: Upload and load the data**

Now we load the **German Credit** dataset, 1,000 real loan applicants, each already labelled a **good** (1) or **bad** (0) credit risk.

In this case, the bank really cared about *creditworthiness*, which nobody can measure directly. The label we actually have, `credit risk`, is a 1994 German bank's "good/bad" rating of applicants. There is no clear record on how it was decided (repayment? an officer's call?).

Download `GermanCredit.csv` from the course site, run the cell, and choose the file when prompted.

In [2]:
df = pd.read_csv("data/GermanCredit.csv")
df.head()

,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,... < 100 DM,6,critical account/other credits existing,domestic appliances,1169,unknown/no savings account,... >= 7 years,4,male : single,none,...,real estate,67,none,own,2,skilled employee/official,1,yes,yes,1
1,0 <= ... < 200 DM,48,existing credits paid back duly till now,domestic appliances,5951,... < 100 DM,1 <= ... < 4 years,2,female : divorced/separated/married,none,...,real estate,22,none,own,1,skilled employee/official,1,no,yes,0
2,no checking account,12,critical account/other credits existing,retraining,2096,... < 100 DM,4 <= ... < 7 years,2,male : single,none,...,real estate,49,none,own,1,unskilled - resident,2,no,yes,1
3,... < 100 DM,42,existing credits paid back duly till now,radio/television,7882,... < 100 DM,4 <= ... < 7 years,2,male : single,guarantor,...,building society savings agreement/life insurance,45,none,for free,1,skilled employee/official,2,no,yes,1
4,... < 100 DM,24,delay in paying off in the past,car (new),4870,... < 100 DM,1 <= ... < 4 years,3,male : single,none,...,unknown/no property,53,none,for free,2,skilled employee/official,2,no,yes,0


In [3]:
df.shape

(1000, 21)

#### **Configuration**

The settings below tell the rest of the notebook what to predict and how to split people into groups. They're filled in for German Credit. To use another dataset, change only this cell - and check two things first.

**1. Which outcome is the "positive" one.**

`POSITIVE_LABEL` marks the favourable decision (here, approve/good risk). All the rates below are read against it. Watch the direction: a default, fraud or re-arrest column usually marks the bad event as 1, so the approve decision is 0. Pick the wrong one and every result reads backwards while the code still runs fine. Set it to match the favourable decision.

**2. How the protected attribute is recorded.**

It's rarely clean. In this dataset, sex and marital status share one column (personal_status_sex), so we pull sex out of it. Before you split by any attribute, check:
- Is it ready to use, or buried in another column you have to decode?
- How is it coded - text or numbers (e.g., 1/2)? Are there undocumented or missing values?
- Is it self-reported, or guessed from something like a name or postcode?
- Are the categories fine enough for the groups you care about?

In [4]:
# --- CONFIGURATION (change only this cell for a different dataset) ---
TARGET_COLUMN = "credit_risk" # 1 = good risk, 0 = bad risk
POSITIVE_LABEL = 1            # a positive decision = "approve/good"
SEX_SOURCE_COLUMN = "personal_status_sex"
AGE_COLUMN = "age"
AGE_THRESHOLD = 25
# ---------------------------------------------------------------------

y = (df[TARGET_COLUMN] == POSITIVE_LABEL).astype(int)
is_male = df[SEX_SOURCE_COLUMN].str.startswith("male")
df["sex"] = is_male.map({True: "male", False: "female"})

df["age_group"] = np.where(df[AGE_COLUMN] < AGE_THRESHOLD, "young (<25)", "older (>=25)")

A = df["sex"]
print("Protected groups (sex):")
print(A.value_counts())

Protected groups (sex):
sex
male      690
female    310
Name: count, dtype: int64


> **What this means**: notice the groups are different sizes, far more men than women. Keep that in mind: the smaller group's numbers will be shakier. You will see that in Step 4.

#### **A note on privacy**

Fairness creates a privacy tension: **to check a model for bias, you have to keep the attributes that are most privacy-sensitive**. You can't audit fairness without sex, age and the like, so they can't just be deleted. Four risks, each with its source and what to do:
1. **Keeping protected attributes to audit.** The audit needs them. *Fix:* keep the attributes for *auditing* separate from the features used to *predict*, and restrict access.

2. **Re-identification.** The data, with no name, age + sex + education + credit limit + repayment pattern can still single a person out. De-identified is not anonymous. *Fix:* treat the records as personal data: aggregate or suppress small subgroups before reporting.

3. **The attribute leaks through other columns.** Dropping sex doesn't remove it if other columns encode it ("fairness through unawareness fails"). *Fix:* assume the signal is still there; deletion alone isn't privacy.

4. **Secondary use.** Reusing account data to train an approval model is a purpose the people never agreed to. *Fix:* lawful basis, notice, and retention limits (DISR *Privacy protection & security*).

---

> **NOTE: don't overstate it**: for a simple model on tabular data, attacks that recover whether someone was in the training set are a minor concern. They matter for large models and foundation models.

---

**Base rates** = how often each group is *actually* a good credit risk in the data. This matters a lot: when the two base rates differ, it becomes mathematically impossible to make every fairness measure equal at once (you'll see this in Step 6).

In [5]:
base_rates = pd.Series(y.values,index=A.values).groupby(level=0).mean().round(3)
print("Base rate(proportion good) by group:")
print(base_rates)

Base rate(proportion good) by group:
female    0.648
male      0.723
dtype: float64


#### **Step 2: Train an ordinary baseline model**

We train a completely standard model, nothing fairness-aware. It scales the numbers, turns text categories into numbers (one-hot encoding), and fits a logistic regression. We split the data into a training part (to learn from) and a test part (to evaluate on), so we judge the model on data it hasn't seen.

The point here is to see how an ordinary model treats different groups.

In [6]:
drop_cols = [TARGET_COLUMN, SEX_SOURCE_COLUMN, "sex", "age_group"]
X = df.drop(columns=drop_cols)

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()

preprocess = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])
baseline=Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(X, y, A, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)
print("Overall accuracy:", round(accuracy_score(y_test, y_pred), 3))

Overall accuracy: 0.733


> **What this means:** that one accuracy number is the trap. It says the model is right about three-quarters of the time *on average*, and tells you nothing about *who* it's right or wrong for. Fairness lives in the breakdown, which we do next.

### **Step 3: Break the results down by group**

`MetricFrame` is Fairlearn's main tool. You hand it your metrics, the true answers, the model's predictions, and the group each person belongs to, and it computes every metric *separately for each group*.

Quick reminder of the rates meaning in this dataset's context:

- **selection_rate**: how often the model says "approve"
- **TPR**: of the people who are truly good risks, how many it correctly approves
- **FPR**: of the people who are truly bad risks, how many it wrongly approves
- **FNR**: of the people who are truly good risks, how many it wrongly rejects